本 notebook 是 [M05.4 训练与优化技巧](../notes/04-training-techniques.qmd) 的实例讲解。

1. 1. 学习率调度的时间单位
2. 2. BatchNorm 与 LayerNorm 的统计轴
3. 3. Dropout 与参数 EMA
4. 4. CPU 混合精度与正确裁剪顺序
5. 5. NaN 的局部定位与安全计算

在 VS Code 中选择仓库 `.venv` 内核，逐单元运行；运行结果随文件保存。示例使用固定随机种子和合成数据。

运行前安装已声明依赖：`uv sync --locked --extra perf --extra data --extra dl`。本例使用 CPU，不需外部数据或预训练权重。

In [1]:
import copy

import numpy as np
import pandas as pd
import torch
from matplotlib_inline.backend_inline import set_matplotlib_formats
from torch import nn

from semiai.plotting import use_style

use_style()
set_matplotlib_formats("svg")
rng = np.random.default_rng(42)

## 1. 学习率调度的时间单位

对实际 SGD 优化器推进 warmup–cosine 调度，记录每次更新使用的学习率，避免 epoch 与 step 混用。

In [2]:
torch.set_num_threads(1)
torch.manual_seed(42)
p = nn.Parameter(torch.tensor(2.0))
optimizer = torch.optim.SGD([p], lr=0.1)


def factor(step):
    if step < 10:
        return (step + 1) / 10
    return 0.5 * (1 + np.cos(np.pi * min((step - 10) / 90, 1.0)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=factor)
records = []
for step in range(100):
    used = optimizer.param_groups[0]["lr"]
    optimizer.zero_grad(set_to_none=True)
    loss = 0.5 * p * p
    loss.backward()
    optimizer.step()
    scheduler.step()
    if step in [0, 9, 10, 50, 99]:
        records.append([step, used, p.item(), loss.item()])
print(
    pd.DataFrame(
        records, columns=["update", "used lr", "parameter after update", "loss before update"]
    )
)

   update   used lr  parameter after update  loss before update
0       0  0.010000                1.980000            2.000000
1       9  0.100000                1.130682            0.789161
2      10  0.100000                1.017613            0.639220
3      50  0.058682                0.029346            0.000486
4      99  0.000030                0.009965            0.000050


读法：记录实际被 optimizer 使用的值，比只看调度公式更能检查首步和恢复偏移；调度器推进在更新之后，ReduceLROnPlateau 等按指标调度的接口另有约定。

## 2. BatchNorm 与 LayerNorm 的统计轴

对固定小张量比较训练期的归一化轴和 running buffer，再检验同一点的 BatchNorm 输出依赖同批样本。

In [3]:
x = torch.tensor([[1.0, 2.0, 3.0], [3.0, 5.0, 7.0], [2.0, 4.0, 8.0], [4.0, 6.0, 9.0]])
bn = nn.BatchNorm1d(3, affine=False, momentum=0.1)
ln = nn.LayerNorm(3, elementwise_affine=False)
y = bn(x)
z = ln(x)
print("BN 各特征 batch 均值/方差:", y.mean(0), y.var(0, unbiased=False))
print("LN 每行均值/方差:", z.mean(1), z.var(1, unbiased=False))
print("BN running mean/variance:", bn.running_mean, bn.running_var)
bn.train()
a = bn(x)[0].clone()
changed = x.clone()
changed[1:] += 10
b = bn(changed)[0]
print("同一首行不同 batch 的 BN 输出:", a, b)
assert torch.allclose(y.mean(0), torch.zeros(3), atol=1e-6)
assert torch.allclose(z.mean(1), torch.zeros(4), atol=1e-6)

BN 各特征 batch 均值/方差: tensor([ 8.9407e-08,  5.9605e-08, -2.9802e-08]) tensor([1.0000, 1.0000, 1.0000])
LN 每行均值/方差: tensor([ 0.0000e+00,  0.0000e+00,  1.1921e-07, -7.9473e-08]) tensor([1.0000, 1.0000, 1.0000, 1.0000])
BN running mean/variance: tensor([0.2500, 0.4250, 0.6750]) tensor([1.0667, 1.1917, 1.5917])
同一首行不同 batch 的 BN 输出: tensor([-1.3416, -1.5213, -1.6465]) tensor([-1.7162, -1.7185, -1.7219])


读法：BN 对 batch 轴聚合，LN 对指定特征轴聚合；BN 训练归一化使用有偏方差，running variance 更新采用无偏估计约定。eval 默认使用运行统计，小 batch 与分布变化都需检查。

## 3. Dropout 与参数 EMA

先核验 inverted Dropout 的训练期期望，再在一个小回归过程上维护参数指数平均，分别报告结果，不假设 EMA 必然更好。

In [4]:
x = torch.ones(20000)
dropout = nn.Dropout(0.4)
sample = dropout(x)
print("Dropout 训练平均:", sample.mean().item(), "零比例:", (sample == 0).float().mean().item())
dropout.eval()
assert torch.equal(dropout(x), x)
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(1, 12), nn.Tanh(), nn.Linear(12, 1))
ema = copy.deepcopy(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.02, weight_decay=0.01)
x = torch.linspace(-1, 1, 80)[:, None]
y = torch.sin(np.pi * x)
for _ in range(150):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.mse_loss(model(x), y)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        for averaged, current in zip(ema.parameters(), model.parameters(), strict=True):
            averaged.mul_(0.95).add_(current, alpha=0.05)
with torch.no_grad():
    print(
        "当前/EMA 训练网格 MSE:",
        nn.functional.mse_loss(model(x), y).item(),
        nn.functional.mse_loss(ema(x), y).item(),
    )

Dropout 训练平均: 0.9960001111030579 零比例: 0.4023999869823456
当前/EMA 训练网格 MSE: 0.06342955678701401 0.08541649580001831


读法：inverted Dropout 在训练时除以保留概率，保持激活期望而非整个非线性网络预测不变；EMA 平滑参数轨迹，但存在启动滞后，含 BN 时 buffer 还需单独处理。

## 4. CPU 混合精度与正确裁剪顺序

用 CPU bfloat16 autocast 实际计算前向与损失；展示 unscale、clip、step 的顺序，GradScaler 在此关闭，不声称验证 CUDA float16。

In [5]:
torch.manual_seed(42)
model = nn.Linear(16, 4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(64, 16)
y = torch.randn(64, 4)
with torch.no_grad():
    reference = model(x)
optimizer.zero_grad(set_to_none=True)
scaler = torch.amp.GradScaler("cpu", enabled=False)
with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    prediction = model(x)
    loss = nn.functional.mse_loss(prediction, y)
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
pre_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5, error_if_nonfinite=True)
post_norm = torch.sqrt(sum((p.grad * p.grad).sum() for p in model.parameters()))
scaler.step(optimizer)
scaler.update()
print("autocast 输出 dtype:", prediction.dtype, "loss dtype:", loss.dtype)
print("与 FP32 前向最大差:", (prediction.float() - reference).abs().max().item())
print("裁剪前/后范数:", pre_norm.item(), post_norm.item(), "scaler enabled:", scaler.is_enabled())
assert post_norm <= 0.50001 and torch.isfinite(loss)

autocast 输出 dtype: torch.bfloat16 loss dtype: torch.float32
与 FP32 前向最大差: 0.00379180908203125
裁剪前/后范数: 0.6575957536697388 0.49999919533729553 scaler enabled: False


读法：bfloat16 与 float16 的指数范围不同；混合精度需按实际算子策略理解。启用缩放时必须先 unscale 再裁剪，不能把缩放后的范数当真实梯度；科学导数核验仍优先双精度。

## 5. NaN 的局部定位与安全计算

展示事后 where 屏蔽无穷值仍可能留下 NaN 梯度，再把无效除法在执行之前消除。

In [6]:
x = torch.tensor([0.0, 1.0], requires_grad=True)
raw = 1 / x
loss = torch.where(x != 0, raw, torch.zeros_like(raw)).sum()
loss.backward()
print("事后屏蔽的梯度:", x.grad)
z = torch.tensor([0.0, 1.0], requires_grad=True)
mask = z != 0
safe_denominator = torch.where(mask, z, torch.ones_like(z))
safe = torch.where(mask, 1 / safe_denominator, torch.zeros_like(z))
safe.sum().backward()
print("运算前处理后的梯度:", z.grad)
assert torch.isfinite(z.grad).all()

事后屏蔽的梯度: tensor([nan, -1.])
运算前处理后的梯度: tensor([ 0., -1.])


读法：前向结果有限不代表反向图每条路径有限；应定位第一个无效运算，而不是仅在最终 loss 上 nan_to_num。裁剪也不能修复已经 NaN 的梯度。

## 小结

训练技巧应对应可观察的机制。先有可运行、可恢复的小循环，再控制学习率、统计尺度、随机正则和精度；通过梯度、激活和独立评价判断改动。诊断顺序越清楚，越不需要依赖同时堆叠许多设置来碰运气。